# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*' 'pyyaml==6.0.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

    # directly download the prepared dataset and shared helpers
    # (Colab only ever gets the 'small' set -- 'full' is local-only, see 1-preparation.ipynb)
    ! mkdir -p 'results'
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-image.zip
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-labels.csv
    ! wget -nv https://raw.githubusercontent.com/mgmalheiros/vision/master/process/counting/common.py
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL,pandas,scipy,yaml'))

# Loading

In [ ]:
# 'small' = the 100-image demo set tracked in the repo. 'full' = the 6021-image
# dataset prepared locally by 1-preparation.ipynb -- see the note there. Only
# 'small' exists on Colab, so leave this as 'small' unless running locally.
dataset_size = 'full'  # 'tiny' or 'full'

import common

images = common.load_prepared_images(base_folder / 'prepared' / f'1-coins-{dataset_size}-image.zip')
df = common.load_labels(base_folder / 'prepared' / f'1-coins-{dataset_size}-labels.csv')

print(f'{len(images)} images loaded')
df.describe()

# Hysteresis Threshold + Morphology Method
Take the Sobel gradient magnitude, threshold it with two levels (hysteresis: a pixel
survives if it is above `high`, or above `low` and connected to one that is), then
clean the resulting mask up with `remove_small_objects` &rarr; fill-holes &rarr;
closing before labeling. Unlike the other notebooks, filtering happens on the mask
itself (`remove_small_objects`), so counting is a plain `label(...).max()`.

In [ ]:
from scipy.ndimage import binary_fill_holes
from skimage import filters, measure, morphology
from skimage.color import label2rgb

In [ ]:
def detect_hysteresis(image, low=0.09, high=0.20, min_size=300, closing_disk=5):
    edges = filters.sobel(image)
    hyst = filters.apply_hysteresis_threshold(edges, low, high)

    binary = hyst > 0
    binary = morphology.remove_small_objects(binary, min_size=min_size)
    binary = binary_fill_holes(binary)
    binary = morphology.binary_closing(binary, morphology.disk(closing_disk))

    labeled = measure.label(binary)
    return int(labeled.max()), labeled

## Visual check

In [ ]:
for name in sorted(images)[:3]:
    image = images[name]
    gt = common.real_count(df, name)

    edges = filters.sobel(image)
    hyst = filters.apply_hysteresis_threshold(edges, low=0.09, high=0.20)
    common.P(image, 'original', size=6, cmap='gray')
    common.P(hyst, 'hysteresis threshold', size=6, cmap='magma')

    count, labeled = detect_hysteresis(image)
    result = label2rgb(labeled, image=image)
    common.P(result, f'Real: {gt} | Found: {count}', size=6)
    common.S()

# Evaluation
Run the detector over the whole prepared dataset, score it against `real_count`, and
write a summary to `results/hysteresis_results.yaml` (or `hysteresis_results-full.yaml`
when `dataset_size == 'full'`).

In [ ]:
results, summary = common.evaluate_method(
    images, df,
    lambda image: detect_hysteresis(image)[0],
    method_name='Hysteresis threshold + morphology',
    parameters={'low': 0.09, 'high': 0.20, 'min_size': 300, 'closing_disk': 5},
    results_path=base_folder / 'results' / f'hysteresis_results{"" if dataset_size == "small" else "-full"}.yaml',
)

for key, value in summary.items():
    print(f'{key}: {value}')

In [ ]:
results[['abs_error', 'time_seconds', 'peak_memory_kb']].describe()